# v18 監査ログ分析ノートブック

YAML形式の時刻別監査ログを分析し、パターン検出・処理フローの可視化を行います。

- 日別のリクエスト数・成功率
- Intent型とComplexityレベルの分布
- ロール別パフォーマンス
- パイプライン処理時間の統計
- エラーパターン分析

In [ ]:
import os
import pandas as pd
import yaml
from pathlib import Path
from datetime import datetime, timedelta
import json

# 監査ログディレクトリ
AUDIT_DIR = Path('/mnt/Bushidan-Multi-Agent/audit/v18')

print(f"監査ログディレクトリ: {AUDIT_DIR}")
print(f"存在: {AUDIT_DIR.exists()}")

In [ ]:
# YAML ファイルを読み込んでリスト化
def load_audit_logs(days=7):
    """直近 N 日分の監査ログを読み込む"""
    entries = []
    cutoff = datetime.now() - timedelta(days=days)
    
    if not AUDIT_DIR.exists():
        return pd.DataFrame()
    
    for day_dir in sorted(AUDIT_DIR.iterdir()):
        if not day_dir.is_dir():
            continue
        
        try:
            day_date = datetime.strptime(day_dir.name, '%Y-%m-%d')
            if day_date < cutoff:
                continue
        except ValueError:
            continue
        
        for yaml_file in sorted(day_dir.glob('bushidan-hour-*.yaml')):
            try:
                with open(yaml_file, 'r', encoding='utf-8') as f:
                    content = f.read()
                    # 簡易 YAML パース (entries セクションのみ)
                    if 'entries:' in content:
                        lines = content.split('\n')
                        in_entries = False
                        current_entry = {}
                        
                        for line in lines:
                            if line.strip() == 'entries:':
                                in_entries = True
                                continue
                            
                            if in_entries:
                                if line.startswith('- '):
                                    if current_entry:
                                        entries.append(current_entry)
                                    current_entry = {}
                                elif line.strip().startswith('-') and ':' in line:
                                    # パース開始
                                    key, val = line.strip()[2:].split(':', 1)
                                    current_entry[key.strip()] = val.strip().strip('"')
                                elif ':' in line and in_entries:
                                    key, val = line.strip().split(':', 1)
                                    try:
                                        # 数値の場合は変換
                                        val_clean = val.strip().strip('"')
                                        if val_clean.replace('.', '', 1).isdigit():
                                            current_entry[key.strip()] = float(val_clean)
                                        else:
                                            current_entry[key.strip()] = val_clean
                                    except:
                                        current_entry[key.strip()] = val.strip().strip('"')
                        
                        if current_entry:
                            entries.append(current_entry)
            except Exception as e:
                print(f"エラー {yaml_file.name}: {e}")
    
    return pd.DataFrame(entries)

df = load_audit_logs(days=7)
print(f"\n読み込み件数: {len(df)}")
print(f"\nカラム: {df.columns.tolist()}")
print(f"\n最初の5行:")
df.head()

In [ ]:
# 基本統計
if len(df) > 0:
    print("=== 基本統計 ===")
    print(f"総リクエスト数: {len(df)}")
    print(f"成功率: {df['success'].astype(float).mean() * 100:.1f}%")
    print(f"\n平均実行時間: {df['execution_time_ms'].astype(float).mean():.0f}ms")
    print(f"パイプライン平均時間: {df['pipeline_ms'].astype(float).mean():.0f}ms")
    
    print(f"\nIntent 型分布:")
    print(df['intent_type'].value_counts())
    
    print(f"\nComplexity 分布:")
    print(df['complexity'].value_counts())
    
    print(f"\nSelectedRole 分布:")
    print(df['selected_role'].value_counts())
else:
    print("データが見つかりません。チャットを実行して監査ログを生成してください。")

In [ ]:
# 実行時間の分布
import matplotlib.pyplot as plt

if len(df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 実行時間分布
    df['execution_time_ms'].astype(float).hist(bins=30, ax=axes[0, 0], edgecolor='black')
    axes[0, 0].set_title('実行時間分布 (ms)')
    axes[0, 0].set_xlabel('実行時間 (ms)')
    axes[0, 0].set_ylabel('頻度')
    
    # Intent型別実行時間
    df.groupby('intent_type')['execution_time_ms'].astype(float).mean().plot(
        kind='bar', ax=axes[0, 1], color='steelblue'
    )
    axes[0, 1].set_title('Intent型別 平均実行時間')
    axes[0, 1].set_ylabel('平均実行時間 (ms)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Notion スコア分布
    df['notion_score'].astype(float).hist(bins=20, ax=axes[1, 0], edgecolor='black')
    axes[1, 0].set_title('Notion スコア分布')
    axes[1, 0].set_xlabel('スコア')
    axes[1, 0].set_ylabel('頻度')
    
    # ロール別リクエスト数
    df['selected_role'].value_counts().plot(kind='barh', ax=axes[1, 1], color='coral')
    axes[1, 1].set_title('ロール別リクエスト数')
    axes[1, 1].set_xlabel('リクエスト数')
    
    plt.tight_layout()
    plt.show()
    
    print("✅ グラフ表示完了")

In [ ]:
# パイプライン処理段階別の分析
if len(df) > 0 and 'stage' in df.columns:
    print("=== パイプライン段階分析 ===")
    stage_counts = df['stage'].value_counts()
    print(f"\n処理段階の分布:")
    for stage, count in stage_counts.head(10).items():
        print(f"  {stage}: {count}件 ({count/len(df)*100:.1f}%)")
    
    # 段階別実行時間
    if 'pipeline_ms' in df.columns:
        print(f"\nパイプライン処理時間統計:")
        print(df['pipeline_ms'].astype(float).describe())

In [ ]:
# エラー分析
if len(df) > 0:
    errors = df[df['success'] == False] if 'success' in df.columns else pd.DataFrame()
    
    if len(errors) > 0:
        print(f"=== エラー分析 ===")
        print(f"エラー件数: {len(errors)} / {len(df)} ({len(errors)/len(df)*100:.1f}%)")
        
        if 'error' in errors.columns:
            print(f"\nエラーメッセージ:")
            for idx, error_msg in enumerate(errors['error'].unique()[:5], 1):
                count = (errors['error'] == error_msg).sum()
                print(f"  {idx}. {error_msg[:60]}... ({count}件)")
    else:
        print("✅ エラーなし - 全リクエスト成功")